# Text Cleaning

In [ ]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Setup resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

exclude = string.punctuation
lemmatizer = WordNetLemmatizer()

# Your slang_map
slang_map = {
    "dont": "do not", "im": "i am", "doesnt": "does not", "didnt": "did not",
    "thats": "that is", "ive": "i have", "isnt": "is not", "theres": "there is",
    "wasnt": "was not", "youre": "you are", "couldnt": "could not", "youll": "you will",
    "theyre": "they are", "wouldnt": "would not", "arent": "are not", "havent": "have not",
    "youve": "you have", "whos": "who is", "werent": "were not", "youd": "you would",
    "hasnt": "has not", "shouldnt": "should not", "hadnt": "had not", "weve": "we have",
    "aint": "am not", "wouldve": "would have", "couldve": "could have", "theyd": "they would",
    "theyve": "they have", "theyll": "they will", "hed": "he would","cant":"can not",
    "tv": "television", "dvd": "digital video disc","vhs": "video home system", "scifi": "science fiction",
    "cgi": "computer generated imagery",
    "imdb": "internet movie database", "hbo": "home box office",
    "cia": "central intelligence agency",
    "wwii": "world war two", "bbc": "british broadcasting corporation",
    "80s": "eighties", "70s": "seventies", "60s": "sixties", "50s": "fifties",
    "40s": "forties", "30s": "thirties", "90s": "nineties"
}

stop_word_list = stopwords.words('english')
# Define words that are actually IMPORTANT for sentiment
keep_words = {'not', 'no', 'never', 'but', 'very', 'too', 'so', 'really', 'however', 'nor'}

# Create your custom list by filtering out the 'keep_words'
custom_stopwords = [word for word in stop_word_list if word not in keep_words]

def preprocess_text(text):

    # 1. Lowercase
    text = text.lower()

    # 2. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # 3. Handle non-ASCII
    text = text.encode("ascii", "ignore").decode()
    text = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', text)

    # 4. Remove punctuation
    for char in exclude:
        text = text.replace(char, '')

    # 5. Handle slang + corrections
    for word, replacement in slang_map.items():
        pattern = r'\b' + word + r'\b'
        text = re.sub(pattern, replacement, text)

    text = re.sub(r'\s+', ' ', text).strip()

    # 6. Handle garbage / normalization
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    words = text.split()
    clean_words = []
    save_list = {'ii', 'iii', 'iv'}

    for w in words:
        if w in ['a', 'i'] or w in save_list:
            clean_words.append(w)
            continue

        is_mash = all(c == w[0] for c in w)
        if len(w) > 1 and not is_mash:
            clean_words.append(w)

    text = " ".join(clean_words)

    # 7. Remove custom stopwords
    words = text.split()
    words = [w for w in words if w not in custom_stopwords]
    text = " ".join(words)

    # 8. Lemmatization (verb-based)
    words = text.split()
    text = " ".join([lemmatizer.lemmatize(w, pos=wordnet.VERB) for w in words])

    return text

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
input_text = """
              A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional 'dream' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terribly well done.
              """

print(preprocess_text(input_text))

wonderful little production film technique very unassuming very oldtimebbc fashion give comfort sometimes discomforting sense realism entire piece actors extremely well choose michael sheen not get polari but voice pat too truly see seamless edit guide reference williams diary entries not well worth watch but terrificly write perform piece masterful production one great master comedy life realism really come home little things fantasy guard rather use traditional dream techniques remain solid disappear play knowledge sense particularly scenes concern orton halliwell set particularly flat halliwells murals decorate every surface terribly well do


# Vectorization(Tokenizing)

In [ ]:
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
# Loading the tokenizer
with open('./models/tokenizer.pickle', 'rb') as handle:
    tokenizer = pickle.load(handle)

In [ ]:
input_text1 = "phil alien one quirky film humor base around <OOV> everything rather actual <OOV> first very odd pretty funny but movie progress not find joke <OOV> funny <OOV> low budget film never problem pretty interest character but eventually lose <OOV> imagine film would appeal stoner currently <OOV> something similar but better try brother another planet"

In [ ]:
sequence  = tokenizer.texts_to_sequences([preprocess_text(input_text)])

# Pad Sequences

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
MAX_LEN= 250 # use SAME maxlen as training
padded = pad_sequences(sequence,
                       padding='pre',
                       truncating='pre',
                       maxlen=MAX_LEN)

In [ ]:
padded.shape

(1, 250)

# Load the Trained model

In [ ]:
from tensorflow.keras.models import load_model

model = load_model('./models/sentiment_rnn_model.keras')

# Prediction

In [ ]:
prediction = model.predict(padded)[0][0]

# Convert to label
if prediction > 0.5:
  print( "Positive")
else:
  print("Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Positive


# Convert it into function

In [ ]:
def predict_sentiment(text):

    # Step 1: Clean text
    cleaned_text = preprocess_text(text)

    # Step 2: Convert to sequence
    sequence = tokenizer.texts_to_sequences([cleaned_text])

    # Step 3: Pad sequence (use SAME maxlen as training)
    padded = pad_sequences(sequence, maxlen=MAX_LEN)

    # Step 4: Predict
    prediction = model.predict(padded)[0][0]

    # Step 5: Convert to label
    if prediction > 0.5:
        return "Positive"
    else:
        return "Negative"

In [ ]:
text = "i will slap you."
predict_sentiment(text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step


'Negative'